<div class="alert alert-block alert-info">

# Part 2: Fingerprint Generation

In the previous notebook, you cleaned bioassay data from AID 743139 to ensure the quality, consistency, and reliability of the dataset before we start to build a model. You found that raw data often contains missing values, duplicates, inconsistent formatting, or mislabeled entries that can lead to incorrect conclusions or poorly performing models. By identifying and resolving these issues at the start of our model design, we created a dataset that accurately reflects the information needed for meaningful interpretation and predictive modeling.

In that notebook you identified all agonists and antagonist molecules and indicated them as active. The dataset also had labeled other molecules as inactive. After cleaning saved that data in a file called AID743139_activity_cids.csv. Since the original SMILES were of unknown origin, you also downloaded all SMILES from PubChem per CID to ensure they were cannonical and saved that in AID743139_cids.csv.

In this short notebook, we will convert our SMILES data to binary encoded fingerprints to serve as structural data input for our binary prediction model.


## Generate MACCS keys from SMILES.

Now that we have our SMILES and Output data in the form of biological activity, we need to generate our binary input that describes chemical structure for model training. The presence of a Key is is encoded as `1` for present and `0` for absent.

In [27]:
# load important libraries 
import pandas as pd
from rdkit import Chem
from rdkit.Chem import MACCSkeys

Read the previous saved SMILES data into a pandas datagframe called df_smiles and print out the first few lines of the dataframe.

In [28]:
df_smiles = pd.read_csv("AID743139_SMILES_cids.csv")
df_smiles.head(10)

,cid,smiles
0,12850184,C(C(=O)[C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O.C(...
1,89753,C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O...
2,9403,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2OC(=O)CCC...
3,13218779,C[C@@]12CC[C@@H](C1(C)C)C[C@H]2OC(=O)CSC#N
4,637566,CC(=CCC/C(=C/CO)/C)C
5,4766,C1=CC=C2C(=C1)C(OS2(=O)=O)(C3=CC=C(C=C3)O)C4=C...
6,3080,C(C(CS)S)O
7,7048801,[C@@H]([C@H](C(=O)O)S)(C(=O)O)S
8,51,C(CC(=O)O)C(=O)C(=O)O
9,66435,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2OP(=O)(O)...


Next we generate MACCS Keys for each CID in the dataframe. We will do so by creating a dictionary called fps (fingerprints) that stores all the CIDs with their generated MACCS keys. 

Recall that MACCS Keys is a 166-bit-long fingerprint, but RDKit generates a 167-bit-long fingerprint. It is because the index of a list/vector in many programming languages (including python) begins at 0. To use the original numbering of the MACCS keys (1-166) (rather than 0-165), the MACCS keys were implemented to be 167-bit-long, with Bit 0 being always zero. 


In [29]:
fps=dict()

for idx, row in df_smiles.iterrows() :
    
    mol = Chem.MolFromSmiles(row.smiles)
    
    if mol == None :
        print("Can't generate MOL object:", "CID", row.cid, row.smiles)
    else:
        fps[row.cid] = [row.cid] + list(MACCSkeys.GenMACCSKeys(mol).ToBitString())

[19:55:12] WARNING: not removing hydrogen atom without neighbors


To see what an example key:value pair of our new dictionary is, we can create a variable and return the first item in the dictionary. We should see the *key* being the CID, and the *value* will be the CID and each of the generated MACCS Keys as a list.


In [30]:
first_entry = next(iter(fps.items()))
print(first_entry)

(12850184, [12850184, '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '1', '0', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '1', '1', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '1', '1', '0', '0', '0', '1', '0', '1', '1', '1', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '1', '1', '1', '1', '0', '1', '0', '1', '0', '0', '0', '0', '1', '0', '1'])


To better visualize the dictionary, we can bring it into a pandas datagframe. To make better sense of the data, we should generate column headings for the data. We will do so by createing a list that begins with the label `cid` and creates labels of `MACCS#` where `#` is the number of the MACCs key generated.

In [31]:
# Generate column names
fpbitnames = []

fpbitnames.append('cid')

for i in range(0,167):   # from MACCS000 to MACCS166
    fpbitnames.append( "maccs" + str(i).zfill(3) )

df_fps = pd.DataFrame.from_dict(fps, orient='index', columns=fpbitnames)

In [32]:
df_fps.head(5)

,cid,maccs000,maccs001,maccs002,maccs003,maccs004,maccs005,maccs006,maccs007,maccs008,...,maccs157,maccs158,maccs159,maccs160,maccs161,maccs162,maccs163,maccs164,maccs165,maccs166
12850184,12850184,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,1,0,1
89753,89753,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,1,0,1
9403,9403,0,0,0,0,0,0,0,0,0,...,1,0,1,1,0,1,1,1,1,0
13218779,13218779,0,0,0,0,0,0,0,0,0,...,1,0,1,1,1,0,1,1,1,0
637566,637566,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,0,0,1,0,0


## Merge activity data and fingerprint information

We now have a new dataframe with CID values and MACCS keys. We also have a stored file that has our activity data associated with each CID. To make this data useful for generating a model, we need to link the MACCS Keys data to the biological data. To do so, we merge these two datasets using CID as the index.

First we need to bring in the activity data as a pandas dataframe.

In [33]:
df_activity = pd.read_csv("AID743139_activity_cids.csv")
df_activity.head(3)  # just to make sure our data is read properly

,cid,activity,PUBCHEM_EXT_DATASOURCE_SMILES
0,12850184.0,0,C(C(=O)[C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O.C(...
1,89753.0,0,C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O...
2,9403.0,0,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2OC(=O)CCC...


Since we don't need the external datasource smiles anymore, we can remove that column.

In [34]:
df_activity = df_activity.drop(df_activity.columns[2], axis=1)
df_activity.head(3)

,cid,activity
0,12850184.0,0
1,89753.0,0
2,9403.0,0


In [35]:
# check to see if we have the same CIDS in the fingerprints
df_fps.head(3)

,cid,maccs000,maccs001,maccs002,maccs003,maccs004,maccs005,maccs006,maccs007,maccs008,...,maccs157,maccs158,maccs159,maccs160,maccs161,maccs162,maccs163,maccs164,maccs165,maccs166
12850184,12850184,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,1,0,1
89753,89753,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,1,0,1
9403,9403,0,0,0,0,0,0,0,0,0,...,1,0,1,1,0,1,1,1,1,0


In [36]:
# create a new combined dataframe using CIDs as the index
df_data = df_activity.join(df_fps.set_index('cid'), on='cid')
df_data.head(3)

,cid,activity,maccs000,maccs001,maccs002,maccs003,maccs004,maccs005,maccs006,maccs007,...,maccs157,maccs158,maccs159,maccs160,maccs161,maccs162,maccs163,maccs164,maccs165,maccs166
0,12850184.0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,1,0,1
1,89753.0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,1,0,1
2,9403.0,0,0,0,0,0,0,0,0,0,...,1,0,1,1,0,1,1,1,1,0


Check to see if there are any CIDs for which the MACCS keys could not be generated.  They need to be removed from **df_data**.

In [37]:
df_data[df_data.isna().any(axis=1)]

,cid,activity,maccs000,maccs001,maccs002,maccs003,maccs004,maccs005,maccs006,maccs007,...,maccs157,maccs158,maccs159,maccs160,maccs161,maccs162,maccs163,maccs164,maccs165,maccs166


In [38]:
len(df_data)

6791

In [39]:
df_data = df_data.dropna()
len(df_data)

6791

Save df_data in CSV for future use.

In [41]:
df_data.to_csv('AID743139_activity_MACCS.csv', index=False) 